# Comparación de modelos de embeddings

Compara rendimiento y calidad de recuperación de 6 modelos de embeddings.

Evalúa cada modelo sobre un corpus fijo en español midiendo:
- **Tiempo de indexación** (encode del corpus completo).
- **Tamaño en memoria** de los embeddings resultantes.
- **Calidad subjetiva**: mejor documento recuperado para 5 consultas usando similitud del coseno.
- **Calidad objetiva**: aciertos top-1 contra el documento esperado de cada consulta, resumidos en un **ranking final**.

Solo requiere `sentence-transformers` (y `numpy`, que es dependencia base); todo lo demás es librería estándar de Python.

In [1]:
from __future__ import annotations

import sys
import time
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

c:\Users\fapc3\OneDrive\Escritorio\infra-personaldata\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Datos de prueba

In [2]:
MODELOS: list[str] = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small",
    "paraphrase-multilingual-mpnet-base-v2",
    "distiluse-base-multilingual-cased-v2",
]

CORPUS: list[str] = [
    "El aumento de las tasas de interés del banco central encareció los créditos hipotecarios este trimestre.",
    "La selección nacional de fútbol clasificó al mundial tras vencer 2-0 en el partido de repechaje.",
    "Los astrónomos detectaron una nueva exoluna orbitando un planeta gaseoso a 500 años luz de la Tierra.",
    "La receta tradicional de la paella valenciana lleva arroz, conejo, pollo, garrofón y azafrán.",
    "El nuevo framework de JavaScript promete reducir el tiempo de renderizado en aplicaciones web complejas.",
    "Las lluvias torrenciales provocaron el desborde de dos ríos y la evacuación de tres comunas rurales.",
    "El museo inaugura una exposición retrospectiva con más de cien obras del pintor surrealista.",
    "Un estudio clínico demostró que dormir menos de seis horas aumenta el riesgo de enfermedades cardiovasculares.",
    "La startup chilena levantó diez millones de dólares para expandir su plataforma de pagos digitales en Latinoamérica.",
    "El senado aprobó en primer trámite la reforma al sistema de pensiones tras una larga negociación.",
    "Los glaciares de la Patagonia han perdido cerca del quince por ciento de su masa en las últimas dos décadas.",
    "El chef recomienda marinar el pescado en jugo de limón durante veinte minutos antes de preparar el ceviche.",
    "La banda anunció una gira por diez ciudades para presentar su nuevo álbum de rock alternativo.",
    "Investigadores desarrollaron una batería de estado sólido que duplica la autonomía de los vehículos eléctricos.",
    "El precio del cobre alcanzó su máximo histórico impulsado por la demanda de la industria tecnológica.",
    "La maratón de Santiago reunió a más de treinta mil corredores en su edición número cuarenta.",
    "Un fallo en el sistema de reservas dejó a cientos de pasajeros varados en el aeropuerto internacional.",
    "La biblioteca municipal amplió su horario nocturno para apoyar a los estudiantes en época de exámenes.",
]

CONSULTAS: list[str] = [
    "¿Qué pasó con los créditos para comprar vivienda?",
    "Avances en baterías para autos eléctricos",
    "¿Cómo se prepara un plato típico con pescado crudo?",
    "Efectos del cambio climático en los hielos del sur",
    "Resultados deportivos de la selección de fútbol",
]

# Índice (0-based) del documento del CORPUS que debería recuperar cada consulta.
DOC_ESPERADO: list[int] = [0, 13, 11, 10, 1]

## Estructuras de resultados

In [3]:
@dataclass
class ResultadoConsulta:
    """Mejor documento recuperado para una consulta."""

    consulta: str
    indice_doc: int
    score: float
    texto: str


@dataclass
class ResultadoModelo:
    """Métricas de evaluación de un modelo de embeddings."""

    nombre: str
    tiempo_indexacion_s: float
    tamano_bytes: int
    dimension: int
    consultas: list[ResultadoConsulta] = field(default_factory=list)

## Evaluación

In [4]:
def cargar_modelo(nombre: str) -> SentenceTransformer:
    """Carga un modelo de sentence-transformers en CPU.

    Args:
        nombre: Identificador del modelo en HuggingFace.

    Returns:
        Modelo SentenceTransformer listo para codificar.
    """
    print(f"  Cargando modelo: {nombre} ...", file=sys.stderr)
    return SentenceTransformer(nombre, device="cpu")


def indexar_corpus(
    modelo: SentenceTransformer, corpus: list[str]
) -> tuple[np.ndarray, float]:
    """Genera los embeddings del corpus midiendo el tiempo de indexación.

    Args:
        modelo: Modelo de embeddings cargado.
        corpus: Lista de textos a codificar.

    Returns:
        Tupla (matriz de embeddings [n_docs x dim], segundos transcurridos).
    """
    inicio = time.perf_counter()
    embeddings = modelo.encode(
        corpus, show_progress_bar=False, convert_to_numpy=True
    )
    duracion = time.perf_counter() - inicio
    return embeddings, duracion


def tamano_en_memoria(embeddings: np.ndarray) -> int:
    """Calcula el peso en bytes de la matriz de embeddings.

    Usa ``nbytes`` (contenido real del buffer) más el overhead del objeto
    ndarray reportado por ``sys.getsizeof``.

    Args:
        embeddings: Matriz de embeddings.

    Returns:
        Tamaño total estimado en bytes.
    """
    overhead = sys.getsizeof(embeddings) - embeddings.nbytes
    return embeddings.nbytes + max(overhead, 0)


def similitud_coseno(matriz: np.ndarray, vector: np.ndarray) -> np.ndarray:
    """Calcula la similitud del coseno entre cada fila de la matriz y un vector.

    Args:
        matriz: Embeddings del corpus, forma (n_docs, dim).
        vector: Embedding de la consulta, forma (dim,).

    Returns:
        Array de scores, forma (n_docs,).
    """
    normas_matriz = np.linalg.norm(matriz, axis=1)
    norma_vector = np.linalg.norm(vector)
    # Evita división por cero en vectores degenerados.
    denominador = np.maximum(normas_matriz * norma_vector, 1e-12)
    return (matriz @ vector) / denominador


def recuperar_mejor(
    modelo: SentenceTransformer,
    embeddings_corpus: np.ndarray,
    corpus: list[str],
    consulta: str,
) -> ResultadoConsulta:
    """Encuentra el documento del corpus más similar a una consulta.

    Args:
        modelo: Modelo de embeddings.
        embeddings_corpus: Matriz de embeddings ya indexada.
        corpus: Textos originales (mismo orden que la matriz).
        consulta: Texto de la consulta.

    Returns:
        Resultado con el índice, score y texto del mejor documento.
    """
    vector = modelo.encode([consulta], show_progress_bar=False, convert_to_numpy=True)[0]
    scores = similitud_coseno(embeddings_corpus, vector)
    mejor = int(np.argmax(scores))
    return ResultadoConsulta(
        consulta=consulta,
        indice_doc=mejor,
        score=float(scores[mejor]),
        texto=corpus[mejor],
    )


def evaluar_modelo(
    nombre: str, corpus: list[str], consultas: list[str]
) -> ResultadoModelo:
    """Evalúa un modelo: indexación, tamaño y recuperación por consulta.

    Args:
        nombre: Identificador del modelo.
        corpus: Textos de prueba.
        consultas: Consultas de recuperación.

    Returns:
        Métricas completas del modelo.
    """
    modelo = cargar_modelo(nombre)
    embeddings, tiempo = indexar_corpus(modelo, corpus)
    resultado = ResultadoModelo(
        nombre=nombre,
        tiempo_indexacion_s=tiempo,
        tamano_bytes=tamano_en_memoria(embeddings),
        dimension=embeddings.shape[1],
    )
    for consulta in consultas:
        resultado.consultas.append(
            recuperar_mejor(modelo, embeddings, corpus, consulta)
        )
    return resultado

## Presentación

In [5]:
def formatear_tamano(n_bytes: int) -> str:
    """Convierte bytes a una representación legible en KB o MB.

    Args:
        n_bytes: Cantidad de bytes.

    Returns:
        Cadena como ``"54.0 KB"`` o ``"1.2 MB"``.
    """
    kb = n_bytes / 1024
    if kb < 1024:
        return f"{kb:.1f} KB"
    return f"{kb / 1024:.2f} MB"


def truncar(texto: str, ancho: int) -> str:
    """Recorta un texto a un ancho fijo agregando puntos suspensivos."""
    return texto if len(texto) <= ancho else texto[: ancho - 1] + "…"


def imprimir_tabla(resultados: list[ResultadoModelo]) -> None:
    """Imprime la tabla comparativa y el detalle de consultas con f-strings.

    Args:
        resultados: Métricas de todos los modelos evaluados.
    """
    ancho_modelo = max(len(r.nombre) for r in resultados) + 2
    n_consultas = len(resultados[0].consultas)

    # --- Tabla resumen: en cada consulta se muestra "#doc (score)" ---
    encabezados = ["Modelo", "Tiempo (s)", "Tamaño", "Dim"] + [
        f"Q{i + 1}" for i in range(n_consultas)
    ]
    anchos = [ancho_modelo, 11, 10, 5] + [12] * n_consultas

    separador = "+" + "+".join("-" * (a + 2) for a in anchos) + "+"
    fila_encabezado = "|" + "|".join(
        f" {h:<{a}} " for h, a in zip(encabezados, anchos)
    ) + "|"

    print("\n== TABLA COMPARATIVA ==")
    print(separador)
    print(fila_encabezado)
    print(separador)
    for r in resultados:
        celdas = [
            f"{r.nombre:<{anchos[0]}}",
            f"{r.tiempo_indexacion_s:>{anchos[1]}.3f}",
            f"{formatear_tamano(r.tamano_bytes):>{anchos[2]}}",
            f"{r.dimension:>{anchos[3]}}",
        ] + [
            f"{f'#{c.indice_doc + 1} ({c.score:.2f})':<{anchos[4 + i]}}"
            for i, c in enumerate(r.consultas)
        ]
        print("|" + "|".join(f" {c} " for c in celdas) + "|")
    print(separador)

    # --- Detalle: texto completo recuperado por cada modelo ---
    print("\n== DETALLE DE CONSULTAS (mejor documento por modelo) ==")
    for i in range(n_consultas):
        print(f"\nQ{i + 1}: {resultados[0].consultas[i].consulta}")
        for r in resultados:
            c = r.consultas[i]
            print(
                f"  {truncar(r.nombre, 40):<40} "
                f"score={c.score:.4f}  doc#{c.indice_doc + 1}: "
                f"{truncar(c.texto, 70)}"
            )

## Ejecución

In [6]:
print(f"Corpus: {len(CORPUS)} documentos | Consultas: {len(CONSULTAS)}")
resultados: list[ResultadoModelo] = []
for nombre in MODELOS:
    print(f"\nEvaluando: {nombre}")
    resultados.append(evaluar_modelo(nombre, CORPUS, CONSULTAS))
    print(f"  Listo en {resultados[-1].tiempo_indexacion_s:.3f} s de indexación")

imprimir_tabla(resultados)

Corpus: 18 documentos | Consultas: 5

Evaluando: all-MiniLM-L6-v2


  Cargando modelo: all-MiniLM-L6-v2 ...
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7664.57it/s]


  Listo en 0.077 s de indexación

Evaluando: BAAI/bge-small-en-v1.5


  Cargando modelo: BAAI/bge-small-en-v1.5 ...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7667.97it/s]


  Listo en 0.187 s de indexación

Evaluando: paraphrase-multilingual-MiniLM-L12-v2


...

  Listo en 0.098 s de indexación

Evaluando: paraphrase-multilingual-mpnet-base-v2


  Cargando modelo: paraphrase-multilingual-mpnet-base-v2 ...
c:\Users\fapc3\OneDrive\Escritorio\infra-personaldata\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fapc3\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.wa

  Listo en 0.556 s de indexación

Evaluando: distiluse-base-multilingual-cased-v2


  Cargando modelo: distiluse-base-multilingual-cased-v2 ...
c:\Users\fapc3\OneDrive\Escritorio\infra-personaldata\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fapc3\.cache\huggingface\hub\models--sentence-transformers--distiluse-base-multilingual-cased-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn

  Listo en 0.196 s de indexación

== TABLA COMPARATIVA ==
+-----------------------------------------+-------------+------------+-------+--------------+--------------+--------------+--------------+--------------+
| Modelo                                  | Tiempo (s)  | Tamaño     | Dim   | Q1           | Q2           | Q3           | Q4           | Q5           |
+-----------------------------------------+-------------+------------+-------+--------------+--------------+--------------+--------------+--------------+
| all-MiniLM-L6-v2                        |       0.077 |    27.1 KB |   384 | #1 (0.58)    | #14 (0.73)   | #16 (0.50)   | #15 (0.44)   | #2 (0.73)    |
| BAAI/bge-small-en-v1.5                  |       0.187 |    27.1 KB |   384 | #1 (0.67)    | #14 (0.83)   | #8 (0.63)    | #6 (0.69)    | #2 (0.84)    |
| paraphrase-multilingual-MiniLM-L12-v2   |       0.138 |    27.1 KB |   384 | #1 (0.44)    | #14 (0.79)   | #12 (0.62)   | #11 (0.40)   | #2 (0.71)    |
| intfloat/multili

## Ranking final: ¿cuál es mejor?

Compara los modelos con una métrica objetiva: **aciertos top-1** (cuántas consultas recuperaron el documento esperado según `DOC_ESPERADO`). Los empates se desempatan por score promedio (más alto = más confianza) y luego por tiempo de indexación (más rápido gana).

In [7]:
def contar_aciertos(resultado: ResultadoModelo, esperados: list[int]) -> int:
    """Cuenta cuántas consultas recuperaron el documento esperado.

    Args:
        resultado: Métricas de un modelo evaluado.
        esperados: Índice del documento correcto para cada consulta.

    Returns:
        Número de aciertos top-1.
    """
    return sum(
        1 for c, esperado in zip(resultado.consultas, esperados)
        if c.indice_doc == esperado
    )


def score_promedio(resultado: ResultadoModelo) -> float:
    """Promedia el score del mejor documento en todas las consultas."""
    return sum(c.score for c in resultado.consultas) / len(resultado.consultas)


def imprimir_ranking(
    resultados: list[ResultadoModelo], esperados: list[int]
) -> None:
    """Imprime el ranking final ordenado de mejor a peor modelo.

    Ordena por aciertos (desc), score promedio (desc) y tiempo (asc).

    Args:
        resultados: Métricas de todos los modelos evaluados.
        esperados: Índice del documento correcto para cada consulta.
    """
    n = len(esperados)
    filas = sorted(
        resultados,
        key=lambda r: (
            -contar_aciertos(r, esperados),
            -score_promedio(r),
            r.tiempo_indexacion_s,
        ),
    )

    ancho_modelo = max(len(r.nombre) for r in filas) + 2
    encabezados = ["Pos", "Modelo", "Aciertos", "Score prom", "Tiempo (s)", "Tamaño", "Dim"]
    anchos = [4, ancho_modelo, 9, 11, 11, 10, 5]

    separador = "+" + "+".join("-" * (a + 2) for a in anchos) + "+"
    fila_encabezado = "|" + "|".join(
        f" {h:<{a}} " for h, a in zip(encabezados, anchos)
    ) + "|"

    print("\n== RANKING FINAL ==")
    print(separador)
    print(fila_encabezado)
    print(separador)
    for pos, r in enumerate(filas, start=1):
        celdas = [
            f"{pos:>{anchos[0]}}",
            f"{r.nombre:<{anchos[1]}}",
            f"{f'{contar_aciertos(r, esperados)}/{n}':>{anchos[2]}}",
            f"{score_promedio(r):>{anchos[3]}.4f}",
            f"{r.tiempo_indexacion_s:>{anchos[4]}.3f}",
            f"{formatear_tamano(r.tamano_bytes):>{anchos[5]}}",
            f"{r.dimension:>{anchos[6]}}",
        ]
        print("|" + "|".join(f" {c} " for c in celdas) + "|")
    print(separador)

    mejor = filas[0]
    print(
        f"\n🏆 Mejor modelo: {mejor.nombre} "
        f"({contar_aciertos(mejor, esperados)}/{n} aciertos, "
        f"score promedio {score_promedio(mejor):.4f}, "
        f"{mejor.tiempo_indexacion_s:.3f} s de indexación)"
    )


imprimir_ranking(resultados, DOC_ESPERADO)


== RANKING FINAL ==
+------+-----------------------------------------+-----------+-------------+-------------+------------+-------+
| Pos  | Modelo                                  | Aciertos  | Score prom  | Tiempo (s)  | Tamaño     | Dim   |
+------+-----------------------------------------+-----------+-------------+-------------+------------+-------+
|    1 | intfloat/multilingual-e5-small          |       5/5 |      0.8664 |       0.098 |    27.1 KB |   384 |
|    2 | paraphrase-multilingual-mpnet-base-v2   |       5/5 |      0.6082 |       0.556 |    54.1 KB |   768 |
|    3 | paraphrase-multilingual-MiniLM-L12-v2   |       5/5 |      0.5908 |       0.138 |    27.1 KB |   384 |
|    4 | distiluse-base-multilingual-cased-v2    |       5/5 |      0.4018 |       0.196 |    36.1 KB |   512 |
|    5 | BAAI/bge-small-en-v1.5                  |       3/5 |      0.7322 |       0.187 |    27.1 KB |   384 |
|    6 | all-MiniLM-L6-v2                        |       3/5 |      0.5953 |       